##### DATA PREPROCESSING

In [1]:
from glob import glob
import pandas as pd
from pathlib import Path
import warnings
import zipfile

warnings.filterwarnings('ignore')

In [2]:
# Extract data into working folders
dataset_path = './datasets'
Path(dataset_path).mkdir(parents=True, exist_ok=True)
zipfile_list = glob('*.zip')
for zip_path in zipfile_list:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)

In [3]:
# Process each file to merge them into a single file (only registered votes from the electoral roll)
datafile_list = glob(f'{dataset_path}/**/*.csv')
filtered_datafile_list = [file for file in datafile_list if 'prisionpreventiva' not in file and 'votoanticipado' not in file]

section_type = {'U':'URBANA', 'M':'MIXTA', 'R':'RURAL'}
dataset_list = []
for datafile in filtered_datafile_list:
    data = pd.read_csv(datafile)
    data['TIPOSEC'] = data['TIPOSEC'].map(section_type)
    data['FELECCION'] = pd.to_datetime(data['FELECCION'], dayfirst=True) if '2024' not in datafile else pd.to_datetime(data['FELECCION'], dayfirst=False)
    for str_col in data.select_dtypes(include=[str]).columns:
        data[str_col] = data[str_col].str.strip().str.upper()
    dataset_list.append(data)

# Merge and clean datasets
final_data = pd.concat(dataset_list)
final_data['DEF'] = final_data['DEF'].replace(['', ' '],'0').astype(int)
final_data['DEL'] = final_data['DEL'].replace(['', ' '],'0').fillna('0').astype(int)
print(f'Dataset shape: {final_data.shape}')

Dataset shape: (54872712, 16)


In [4]:
# Process files from other electoral formats
early_voting_data = pd.read_csv(datafile_list[-1])
early_voting_data['MPIONOM'] = early_voting_data['MPIONOM'].str.strip().str.upper()

pretial_detention_data = pd.read_csv(datafile_list[-1])
pretial_detention_data['MPIONOM'] = pretial_detention_data['MPIONOM'].str.strip().str.upper()

In [ ]:
# Save datasets
output_path = './final_datasets'
Path(output_path).mkdir(parents=True, exist_ok=True)
final_data.to_parquet(f'{output_path}/conteos_censales.parquet')
early_voting_data.to_csv(f'{output_path}/voto_anticipado_2024.csv')
pretial_detention_data.to_csv(f'{output_path}/prision_preventiva_2024.csv')